## Multi head attention class with weight matrix split

In [22]:
import torch
import torch.nn as nn
inputs=torch.tensor([[0.1,0.2,0.3,0.4,0.5,0.6],
                    [0.1,0.2,0.3,0.4,0.5,0.6],
                    [0.1,0.2,0.3,0.4,0.5,0.6],
                    [0.1,0.2,0.3,0.4,0.5,0.6],
                    [0.1,0.2,0.3,0.4,0.5,0.6],
                    [0.1,0.2,0.3,0.4,0.5,0.6]])

inputs=torch.stack((inputs,inputs),dim=0)
inputs.shape[-1]

6

In [23]:
d_in=inputs.shape[-1]
d_out=6
num_heads=2
head_dim=int(d_out/num_heads)
dropout=0.0

inputs.shape

torch.Size([2, 6, 6])

In [24]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
## step -2 decide d_out and num_head

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

## Step- 3 initalize trainable wt

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

## Step - 4 calculate Keys , values , queries
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
## Step -5  Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

## Step -6  Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

## Step -7  Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

## Step-8 attention wt

      # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

## Step -9 Context Vec 
## Step -10 reformate context vec 
#  Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
## Step -11  Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [25]:
b,context_length,d_in=inputs.shape
d_out=2

In [26]:
a=MultiHeadAttention(d_in,d_out,context_length,dropout,num_heads)

In [27]:
a(inputs)

tensor([[[0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165]],

        [[0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165],
         [0.4116, 0.3165]]], grad_fn=<ViewBackward0>)